# Train the Brent crude market proxy

This notebook reads the Brent history from `backend/ml_assets/data/`, runs a chronological holdout against a persistence baseline, and saves the trained model plus evaluation metadata in this `models/` directory. The target is a crude-oil proxy, not marine bunker fuel or an India-route freight rate.

In [ ]:
import json
from datetime import datetime, timezone
from pathlib import Path

import joblib
import numpy as np
import pandas as pd
from sklearn.linear_model import Ridge
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler

cwd = Path.cwd().resolve()
candidates = [parent / 'backend' / 'ml_assets' for parent in (cwd, *cwd.parents)]
candidates += [parent / 'ml_assets' for parent in (cwd, *cwd.parents)]
ASSET_DIR = next(path for path in candidates if (path / 'data' / 'brent-daily.csv').exists())
DATA_PATH = ASSET_DIR / 'data' / 'brent-daily.csv'
MODEL_PATH = ASSET_DIR / 'models' / 'brent_proxy_model.joblib'
RESULTS_PATH = ASSET_DIR / 'models' / 'brent_proxy_evaluation.json'
HORIZON = 5
print('Input:', DATA_PATH)
print('Model output:', MODEL_PATH)

In [ ]:
def make_features(prices):
    frame = pd.DataFrame(index=prices.index)
    for lag in (1, 2, 5, 10, 20, 60):
        frame[f'price_lag_{lag}'] = prices.shift(lag)
    daily_returns = prices.pct_change()
    for window in (5, 20, 60):
        frame[f'return_{window}'] = prices.pct_change(window)
        frame[f'volatility_{window}'] = daily_returns.rolling(window).std()
    frame['year_sin'] = np.sin(2 * np.pi * frame.index.dayofyear / 365.25)
    frame['year_cos'] = np.cos(2 * np.pi * frame.index.dayofyear / 365.25)
    return frame.replace([np.inf, -np.inf], np.nan)

def rmse(actual, predicted):
    return float(np.sqrt(np.mean(np.square(actual - predicted))))

raw = pd.read_csv(DATA_PATH)
prices = pd.Series(
    pd.to_numeric(raw['Price'], errors='coerce').to_numpy(),
    index=pd.to_datetime(raw['Date'], errors='coerce'),
    name='brent_usd_per_barrel',
).dropna().sort_index()
prices = prices.loc[~prices.index.duplicated(keep='last')]
features = make_features(prices)
target = prices.shift(-HORIZON).rename('target')
dataset = features.join(target).dropna()
split = int(len(dataset) * 0.8)
train, holdout = dataset.iloc[:split], dataset.iloc[split:]
print(f'{len(prices):,} observations · {prices.index[0].date()} to {prices.index[-1].date()}')
print(f'{len(train):,} training rows · {len(holdout):,} chronological holdout rows · {HORIZON} observations ahead')

In [ ]:
candidate = make_pipeline(StandardScaler(), Ridge(alpha=100))
candidate.fit(train[features.columns], train['target'])
candidate_predictions = candidate.predict(holdout[features.columns])
persistence_predictions = holdout['price_lag_1'].to_numpy()
actual = holdout['target'].to_numpy()
candidate_rmse = rmse(actual, candidate_predictions)
persistence_rmse = rmse(actual, persistence_predictions)
holdout_mae = float(np.mean(np.abs(actual - candidate_predictions)))
holdout_metrics = {
    'candidate_rmse_usd_per_barrel': round(candidate_rmse, 4),
    'persistence_rmse_usd_per_barrel': round(persistence_rmse, 4),
    'candidate_mae_usd_per_barrel': round(holdout_mae, 4),
    'candidate_beats_persistence': candidate_rmse < persistence_rmse,
}
holdout_metrics

In [ ]:
final_model = make_pipeline(StandardScaler(), Ridge(alpha=100))
final_model.fit(dataset[features.columns], dataset['target'])
latest_date = prices.index[-1]
latest_features = features.loc[[latest_date], features.columns]
bundle = {
    'model': final_model,
    'feature_columns': list(features.columns),
    'horizon_observations': HORIZON,
    'data_as_of': latest_date.date().isoformat(),
    'latest_price': float(prices.loc[latest_date]),
    'predicted_price': float(final_model.predict(latest_features)[0]),
}
joblib.dump(bundle, MODEL_PATH)
evaluation = {
    'status': 'prototype_only',
    'target': 'Brent crude oil spot price proxy, USD per barrel',
    'data_source_file': str(DATA_PATH.relative_to(ASSET_DIR)),
    'data_start': prices.index[0].date().isoformat(),
    'data_as_of': latest_date.date().isoformat(),
    'observations': int(len(prices)),
    'horizon_observations': HORIZON,
    'training_rows': int(len(train)),
    'holdout_rows': int(len(holdout)),
    'holdout': holdout_metrics,
    'model_file': str(MODEL_PATH.relative_to(ASSET_DIR)),
    'trained_at_utc': datetime.now(timezone.utc).isoformat(),
    'recommended_for_decision_use': False,
    'limitations': [
        'Brent crude is a broad fuel-cost proxy, not marine bunker fuel.',
        'This model does not forecast an India-route freight rate or charter quote.',
        'Holdout improvement over persistence is small and does not guarantee future accuracy.',
        'The input CSV ends at data_as_of; no live EIA refresh is configured.',
    ],
}
RESULTS_PATH.write_text(json.dumps(evaluation, indent=2) + '\n')
print(json.dumps({**evaluation, 'predicted_price': round(bundle['predicted_price'], 2)}, indent=2))